In [1]:
import pickle
import numpy as np

In [2]:
dataset = "MIMIC-IV"
finetune_data_path_1 = f"./data_process/{dataset}-processed/mimic_nextdiag_6m.pkl"
finetune_data_path_2 = f"./data_process/{dataset}-processed/mimic_nextdiag_12m.pkl"

In [3]:
train_data, val_data, test_data = pickle.load(open(finetune_data_path_2, 'rb'))

In [4]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Callable, Iterable, Tuple
import pandas as pd

LABEL_ORDER: List[str] = [
    "Acute and unspecified renal failure",
    "Acute cerebrovascular disease",
    "Acute myocardial infarction",
    "Cardiac dysrhythmias",
    "Chronic kidney disease",
    "Chronic obstructive pulmonary disease",
    "Conduction disorders",
    "Congestive heart failure; nonhypertensive",
    "Coronary atherosclerosis and related",
    "Disorders of lipid metabolism",
    "Essential hypertension",
    "Fluid and electrolyte disorders",
    "Gastrointestinal hemorrhage",
    "Hypertension with complications",
    "Other liver diseases",
    "Other lower respiratory disease",
    "Pneumonia",
    "Septicemia (except in labor)",
]

# ---------- 关键更新：去掉 DIAG_ 前缀（大小写不敏感，容忍 DIAG- / DIAG: / 多余空格） ----------
def _norm_icd9(code: str) -> str:
    """
    标准化 ICD-9：去掉前缀 DIAG_ / DIAG- / DIAG:（大小写不敏感），再去空格、转大写。
    例：'diag_584.9' -> '584.9'；'DIAG: 038.9' -> '038.9'
    """
    s = str(code).strip()
    s = re.sub(r'^\s*diag[\s_:\-]*', '', s, flags=re.IGNORECASE)
    return s.strip().upper()

def _icd9_3d(code: str) -> int:
    s = _norm_icd9(code).split('.')[0].lstrip('0')
    return int(s) if s.isdigit() else -1

def _startswith(code: str, prefix: str) -> bool:
    return _norm_icd9(code).startswith(prefix)

def _equals(code: str, exact: str) -> bool:
    return _norm_icd9(code) == exact

def _in_3d_range(code: str, lo: int, hi: int) -> bool:
    x = _icd9_3d(code)
    return lo <= x <= hi

def _any(codes: Iterable[str], pred: Callable[[str], bool]) -> bool:
    return any(pred(c) for c in codes)

def _to_code_list(x):
    """
    将单元格内容规范化为 ICD-9 列表，兼容：
    - np.ndarray(dtype=object)/list/tuple -> 逐元素清洗
    - 字符串（逗号/空白分隔，或 "['DIAG_5849','DIAG_0389']"）-> 拆分并清洗
    - pd.NA/NaN/None -> []
    - 其他标量 -> 单元素列表
    """
    # 空值处理
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if x is pd.NA:
        return []
    # numpy 数组（你的数据就是这种）
    if isinstance(x, np.ndarray):
        # 有时整个单元格就是一个 object 数组；直接逐元素清洗
        return [_norm_icd9(xx) for xx in x.tolist() if pd.notna(xx)]
    # Python 容器
    if isinstance(x, (list, tuple, set)):
        return [_norm_icd9(xx) for xx in x if pd.notna(xx)]
    # 字符串
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.upper() == "NAN":
            return []
        # 去掉外层中括号/括号引号等，再按逗号或空白切
        s = s.strip(" []()")
        parts = re.split(r"[,\s]+", s)
        return [_norm_icd9(p) for p in parts if p and p.upper() != "NAN"]
    # 其他标量
    return [_norm_icd9(x)]

def _mk_labelers():
    return {
        "Acute and unspecified renal failure": lambda xs: _any(xs, lambda c: _startswith(c, "584")),
        "Acute cerebrovascular disease":       lambda xs: _any(xs, lambda c: _in_3d_range(c, 430, 437)),
        "Acute myocardial infarction":         lambda xs: _any(xs, lambda c: _startswith(c, "410")),
        "Cardiac dysrhythmias":                lambda xs: _any(xs, lambda c: _startswith(c, "427")),
        "Chronic kidney disease":              lambda xs: _any(xs, lambda c: _startswith(c, "585")),
        "Chronic obstructive pulmonary disease": lambda xs: _any(xs, lambda c: _in_3d_range(c, 490, 492) or _startswith(c, "494") or _startswith(c, "496")),
        "Conduction disorders":                lambda xs: _any(xs, lambda c: _startswith(c, "426")),
        "Congestive heart failure; nonhypertensive": lambda xs: _any(xs, lambda c: _startswith(c, "428")),
        "Coronary atherosclerosis and related": lambda xs: _any(xs, lambda c: _startswith(c, "414")),
        "Disorders of lipid metabolism":       lambda xs: _any(xs, lambda c: _startswith(c, "272")),
        "Essential hypertension":              lambda xs: _any(xs, lambda c: _startswith(c, "401")),
        "Fluid and electrolyte disorders":     lambda xs: _any(xs, lambda c: _startswith(c, "276")),
        "Gastrointestinal hemorrhage":         lambda xs: _any(xs, lambda c: _startswith(c, "578")),
        "Hypertension with complications":     lambda xs: _any(xs, lambda c: any(_startswith(c, p) for p in ["402","403","404","405"])),
        "Other liver diseases":                lambda xs: _any(xs, lambda c: _in_3d_range(c, 570, 573)),
        "Other lower respiratory disease":     lambda xs: _any(xs, lambda c: (
                                                        _in_3d_range(c, 513, 517) or
                                                        (_startswith(c, "518") and not any(_startswith(c, s) for s in ["518.81","518.82","518.83","518.84"])) or
                                                        _startswith(c, "519")
                                                    )),
        "Pneumonia":                           lambda xs: _any(xs, lambda c: _in_3d_range(c, 480, 486)),
        "Septicemia (except in labor)":        lambda xs: _any(xs, lambda c: _startswith(c, "038") or _equals(c, "785.52") or _equals(c, "995.91") or _equals(c, "995.92")),
    }

def add_multilabel_with_summary(
    df: pd.DataFrame,
    icd9_list_col: str = "next_enc_diagnoses_icd9",
    output_prefix: str = "ML_",
    summary_col: str = "multi_label"
) -> Tuple[pd.DataFrame, List[str]]:
    """
    将 df[icd9_list_col]（每行是 ICD-9 代码列表/字符串，例如 'DIAG_584.9'）转换为：
      - 25 个 0/1 列（列名 f"{output_prefix}{label}"）
      - 1 个汇总列 summary_col：长度 25 的二进制 list（顺序见 LABEL_ORDER）
    返回: (新 DataFrame, LABEL_ORDER)
    """
    if icd9_list_col not in df.columns:
        raise KeyError(f"Column '{icd9_list_col}' not found in df.")

    labelers = _mk_labelers()
    missing = [lab for lab in LABEL_ORDER if lab not in labelers]
    if missing:
        raise ValueError(f"Labelers missing definitions for: {missing}")

    out = df.copy()
    norm_codes_series = out[icd9_list_col].apply(_to_code_list)

    # 逐列打标签（0/1）
    for label_name in LABEL_ORDER:
        pred = labelers[label_name]
        out[f"{output_prefix}{label_name}"] = norm_codes_series.apply(lambda codes: int(bool(pred(codes))))

    # 汇总列
    out[summary_col] = out.apply(
        lambda row: [int(row[f"{output_prefix}{name}"]) for name in LABEL_ORDER],
        axis=1
    )
    print(LABEL_ORDER)
    return out

In [5]:
train_data_out = add_multilabel_with_summary(
    train_data,
    icd9_list_col="NEXT_DIAG_12M",
    output_prefix="PHENO_",
    summary_col="NEXT_DIAG_12M_PHENO"
)

val_data_out = add_multilabel_with_summary(
    val_data,
    icd9_list_col="NEXT_DIAG_12M",
    output_prefix="PHENO_",
    summary_col="NEXT_DIAG_12M_PHENO"
)

test_data_out = add_multilabel_with_summary(
    test_data,
    icd9_list_col="NEXT_DIAG_12M",
    output_prefix="PHENO_",
    summary_col="NEXT_DIAG_12M_PHENO"
)


['Acute and unspecified renal failure', 'Acute cerebrovascular disease', 'Acute myocardial infarction', 'Cardiac dysrhythmias', 'Chronic kidney disease', 'Chronic obstructive pulmonary disease', 'Conduction disorders', 'Congestive heart failure; nonhypertensive', 'Coronary atherosclerosis and related', 'Disorders of lipid metabolism', 'Essential hypertension', 'Fluid and electrolyte disorders', 'Gastrointestinal hemorrhage', 'Hypertension with complications', 'Other liver diseases', 'Other lower respiratory disease', 'Pneumonia', 'Septicemia (except in labor)']
['Acute and unspecified renal failure', 'Acute cerebrovascular disease', 'Acute myocardial infarction', 'Cardiac dysrhythmias', 'Chronic kidney disease', 'Chronic obstructive pulmonary disease', 'Conduction disorders', 'Congestive heart failure; nonhypertensive', 'Coronary atherosclerosis and related', 'Disorders of lipid metabolism', 'Essential hypertension', 'Fluid and electrolyte disorders', 'Gastrointestinal hemorrhage', 'Hy

In [6]:
def print_prevalence(df: pd.DataFrame, prefix: str = "PHENO_") -> pd.DataFrame:
    """
    计算并打印 DataFrame 中以 prefix 开头的多标签列的 prevalence。
    只统计 scalar 0/1 列，不包括 list 类型的列。
    """
    # 找到所有以 prefix 开头 且 dtype 不是 object(list) 的列
    label_cols = [c for c in df.columns if c.startswith(prefix) and df[c].map(lambda x: isinstance(x, (int, float))).all()]
    if not label_cols:
        raise ValueError(f"No valid 0/1 columns start with prefix '{prefix}'.")

    n = len(df)
    prevalences = {col: df[col].sum() / n for col in label_cols}

    prevalence_df = (
        pd.DataFrame({"Disease": label_cols,
                      "Prevalence": [prevalences[c] for c in label_cols]})
        .sort_values("Prevalence", ascending=False)
        .reset_index(drop=True)
    )

    # 打印（百分比格式）
    print("=== Disease Prevalence ===")
    for _, row in prevalence_df.iterrows():
        print(f"{row['Disease']}: {row['Prevalence']*100:.1f}%")

    return prevalence_df

In [7]:
prevalence_df = print_prevalence(train_data_out, prefix="PHENO_")
prevalence_df = print_prevalence(val_data_out, prefix="PHENO_")
prevalence_df = print_prevalence(test_data_out, prefix="PHENO_")

=== Disease Prevalence ===
PHENO_Essential hypertension: 22.6%
PHENO_Disorders of lipid metabolism: 20.9%
PHENO_Fluid and electrolyte disorders: 18.5%
PHENO_Cardiac dysrhythmias: 15.4%
PHENO_Coronary atherosclerosis and related: 13.7%
PHENO_Chronic kidney disease: 13.5%
PHENO_Acute and unspecified renal failure: 13.2%
PHENO_Congestive heart failure; nonhypertensive: 12.8%
PHENO_Hypertension with complications: 11.9%
PHENO_Other lower respiratory disease: 9.1%
PHENO_Septicemia (except in labor): 5.2%
PHENO_Chronic obstructive pulmonary disease: 4.5%
PHENO_Pneumonia: 4.2%
PHENO_Gastrointestinal hemorrhage: 2.1%
PHENO_Acute myocardial infarction: 1.8%
PHENO_Conduction disorders: 1.2%
PHENO_Other liver diseases: 0.8%
PHENO_Acute cerebrovascular disease: 0.5%
=== Disease Prevalence ===
PHENO_Essential hypertension: 22.6%
PHENO_Disorders of lipid metabolism: 20.3%
PHENO_Fluid and electrolyte disorders: 17.4%
PHENO_Cardiac dysrhythmias: 14.7%
PHENO_Coronary atherosclerosis and related: 13.1%


In [8]:
with open(finetune_data_path_2, "wb") as outfile:
    pickle.dump([train_data_out, val_data_out, test_data_out], outfile)

In [53]:
train_data_out

,SUBJECT_ID,HADM_ID,ICD9_CODE,NDC,PRO_CODE,LAB_TEST,STAY_DAYS,GENDER,AGE,DEATH,...,PHENO_Disorders of lipid metabolism,PHENO_Essential hypertension,PHENO_Fluid and electrolyte disorders,PHENO_Gastrointestinal hemorrhage,PHENO_Hypertension with complications,PHENO_Other liver diseases,PHENO_Other lower respiratory disease,PHENO_Pneumonia,PHENO_Septicemia (except in labor),NEXT_DIAG_12M_PHENO
81,124,172461,"[DIAG_43331, DIAG_99812, DIAG_436, DIAG_43320,...","[MED_N01A, MED_A02B]","[PRO_3812, PRO_3950, PRO_3990, PRO_8841]","[LAB_51288-3.0, LAB_51237-0.0, LAB_51274-1.0, ...",20,M,AGE_15,False,...,0,0,0,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
82,124,134369,"[DIAG_44021, DIAG_5849, DIAG_5854, DIAG_5990, ...","[MED_A12C, MED_C02D, MED_A02B, MED_C07A, MED_R...","[PRO_3950, PRO_3818, PRO_3957, PRO_3990, PRO_0...","[LAB_51476-0.0, LAB_51491-0.0, LAB_51492-1.0, ...",15,M,AGE_16,False,...,0,0,0,0,1,0,0,1,1,"[1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, ..."
83,124,138376,"[DIAG_56983, DIAG_56089, DIAG_49121, DIAG_9985...","[MED_N02A, MED_C07A, MED_A12A, MED_A07A]","[PRO_4573, PRO_311, PRO_4621, PRO_5472, PRO_33...","[LAB_50802-0.0, LAB_50804-0.0, LAB_50808-0.0, ...",31,M,AGE_16,True,...,0,0,0,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
109,156,168847,"[DIAG_44103, DIAG_48241, DIAG_51881, DIAG_4280...","[MED_B05C, MED_C02D, MED_N05B, MED_A12C, MED_A...","[PRO_9604, PRO_9672, PRO_966, PRO_3893, PRO_3891]","[LAB_50820-0.0, LAB_50821-1.0, LAB_50825-3.0, ...",20,M,AGE_13,False,...,0,0,0,0,0,0,1,1,0,"[1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
110,156,199280,"[DIAG_44103, DIAG_5185, DIAG_5109, DIAG_99811,...","[MED_B05C, MED_A12A, MED_D07A, MED_N02A, MED_A...","[PRO_3845, PRO_311, PRO_3844, PRO_0013, PRO_00...","[LAB_50822-4.0, LAB_50824-4.0, LAB_50809-0.0, ...",27,M,AGE_13,False,...,0,0,0,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39859,99538,128902,"[DIAG_42823, DIAG_5119, DIAG_E8782, DIAG_4280,...","[MED_S01E, MED_C03C, MED_A07E, MED_A12A]","[PRO_9604, PRO_9671, PRO_3491, PRO_3891, PRO_3...",[LAB_50803-4.0],3,F,AGE_16,False,...,0,0,0,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
39869,99562,129689,"[DIAG_99659, DIAG_53240, DIAG_5762, DIAG_42833...","[MED_A07A, MED_C03C, MED_J01M, MED_J01C, MED_H...","[PRO_5110, PRO_9705, PRO_3995]","[LAB_51143-0.0, LAB_51144-0.0, LAB_51251-0.0, ...",15,M,AGE_13,False,...,1,0,1,0,1,0,1,0,0,"[1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, ..."
39870,99562,101705,"[DIAG_42821, DIAG_41071, DIAG_41412, DIAG_5845...","[MED_N02B, MED_C03C, MED_C05A, MED_C01B, MED_C...","[PRO_0066, PRO_3606, PRO_9672, PRO_8856, PRO_3...","[LAB_51078-3.0, LAB_51082-0.0, LAB_51093-1.0, ...",15,M,AGE_13,False,...,0,0,0,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
39916,99783,174582,"[DIAG_7464, DIAG_4412, DIAG_2859, DIAG_4263, D...","[MED_J01D, MED_A12A, MED_A12C, MED_A10A, MED_A...","[PRO_3522, PRO_3845, PRO_3961]","[LAB_50810-2.0, LAB_50811-2.0, LAB_50824-0.0, ...",5,M,AGE_9,False,...,0,0,0,0,0,1,1,0,1,"[1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, ..."
